# 长上下文 Needle 评测：位置分桶、干扰项与版本冲突

**面试问题：怎样设计一个能发现 Lost-in-the-Middle，而不是被记忆或字符串泄漏欺骗的评测？**

## 回答主线

1. Needle 评测应把随机键值插入多个长度和相对位置，并用不含答案的查询读取。
2. 每个样本需要同主题干扰项，防止模型只按最近数字或固定模板猜答案。
3. 结果必须按 0–20%、20–40% 等位置桶报告，单一平均准确率会掩盖中部退化。
4. 同时运行 head-tail 截断基线和全量检索基线，可以区分上下文读取问题与数据管道问题。
5. 答案版本冲突时需要有效时间和来源规则，不能把任意命中的旧值算正确。
6. 真实评测还要随机化模板、做负例、阻断训练污染并重复采样置信区间。

## 真实案例

五份 100 段的客服审计文档在 5%、25%、50%、75%、95% 位置插入不同订单退款码，每份另放四个相似干扰码。我们实现可复现数据生成、head-tail 上下文基线和遍历全部分块的关键词检索，按位置桶输出准确率；随后加入同订单旧版本冲突。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：五个位置桶与随机 Needle

In [1]:
positions = [5, 25, 50, 75, 95]  # 定义五个相对位置百分点。
codes = ["RF-A17", "RF-B29", "RF-C43", "RF-D58", "RF-E91"]  # 定义每份文档唯一退款码。
documents = []  # 收集生成的长文档样本。
for sample_index, (position, code) in enumerate(zip(positions, codes), start=1):  # 逐位置生成一百段文档。
    order_id = f"ORDER-{sample_index}"  # 为样本生成不重复订单号。
    paragraphs = [f"第{index:03d}段：客服审计记录正常，未发现特殊指令。" for index in range(100)]  # 构造九十九类背景段落。
    paragraphs[position] = f"权威退款记录：订单 {order_id} 当前退款码为 {code}，版本 2026-07。"  # 在指定位置插入唯一 Needle。
    distractor_positions = [12, 37, 63, 88]  # 固定四个干扰位置便于复现。
    for distractor_index, distractor_position in enumerate(distractor_positions, start=1):  # 写入同模板干扰码。
        if distractor_position != position:  # 避免覆盖真实 Needle。
            paragraphs[distractor_position] = f"其他订单 OTHER-{sample_index}-{distractor_index} 的退款码为 NOISE-{distractor_index}。"  # 添加相似但订单不同的干扰证据。
    documents.append({"id": f"DOC-{sample_index}", "order": order_id, "position": position, "code": code, "paragraphs": paragraphs, "query": f"{order_id} 当前退款码是什么"})  # 保存评测样本。
print("文档   订单      Needle位置  正确码    查询")  # 输出评测输入表头。
for document in documents:  # 逐样本展示位置和目标。
    print(f"{document['id']}  {document['order']:<8} {document['position']:>9}%  {document['code']:<7}  {document['query']}")  # 展示五个位置桶。

文档   订单      Needle位置  正确码    查询
DOC-1  ORDER-1          5%  RF-A17   ORDER-1 当前退款码是什么
DOC-2  ORDER-2         25%  RF-B29   ORDER-2 当前退款码是什么
DOC-3  ORDER-3         50%  RF-C43   ORDER-3 当前退款码是什么
DOC-4  ORDER-4         75%  RF-D58   ORDER-4 当前退款码是什么
DOC-5  ORDER-5         95%  RF-E91   ORDER-5 当前退款码是什么


## Baseline 基线：只把头尾各十段放进上下文

In [2]:
def extract_code(paragraphs, order_id):  # 从可见段落中抽取指定订单的退款码。
    prefix = f"订单 {order_id} 当前退款码为 "  # 构造必须同时匹配订单的权威模式。
    for paragraph in paragraphs:  # 顺序扫描可见文本。
        if prefix in paragraph:  # 当前段包含目标订单而非其他干扰订单。
            remainder = paragraph.split(prefix, 1)[1]  # 截取退款码之后文本。
            return remainder.split("，", 1)[0]  # 返回逗号前的退款码。
    return None  # 上下文中找不到 Needle 时返回空。

baseline_rows = []  # 收集 head-tail 截断结果。
for document in documents:  # 逐长文档评估。
    visible = document["paragraphs"][:10] + document["paragraphs"][-10:]  # 只保留头尾各十段。
    prediction = extract_code(visible, document["order"])  # 尝试从截断上下文读取目标码。
    baseline_rows.append({"id": document["id"], "position": document["position"], "prediction": prediction, "correct": prediction == document["code"]})  # 保存逐样本结果。
print("文档   position  预测      正确")  # 输出截断基线表头。
for row in baseline_rows:  # 逐位置展示 lost-in-the-middle。
    print(f"{row['id']} {row['position']:>8}%  {str(row['prediction']):<9} {row['correct']}")  # 展示只有头尾 Needle 可见。

文档   position  预测      正确
DOC-1        5%  RF-A17    True
DOC-2       25%  None      False
DOC-3       50%  None      False
DOC-4       75%  None      False
DOC-5       95%  RF-E91    True


### 核心实现：全分块检索与逐候选得分

In [3]:
def retrieval_score(query, paragraph, order_id):  # 用订单实体和任务词计算可解释关键词分数。
    score = 0  # 初始化段落相关性。
    score += 5 if order_id in paragraph else 0  # 订单精确匹配是最强信号。
    score += 2 if "退款码" in paragraph else 0  # 退款码任务词提供次强信号。
    score += 1 if "当前" in paragraph else 0  # 当前版本词区分历史记录。
    score += sum(term in paragraph for term in query.split())  # 加入查询原词命中数。
    return score  # 返回可解释总分。

def retrieve(document, top_k=3):  # 遍历全部一百段并返回最高分候选。
    rows = []  # 收集段落位置和得分。
    for index, paragraph in enumerate(document["paragraphs"]):  # 扫描完整长文档。
        score = retrieval_score(document["query"], paragraph, document["order"])  # 计算当前段相关性。
        rows.append({"index": index, "score": score, "text": paragraph})  # 保存候选证据。
    return sorted(rows, key=lambda row: (-row["score"], row["index"]))[:top_k]  # 按分数和位置稳定排序。

demo_candidates = retrieve(documents[2])  # 对正中间 Needle 展示检索过程。
print("50% Needle 的 Top-3 候选：")  # 输出最能暴露中部问题的样本。
for candidate in demo_candidates:  # 逐候选展示位置、分数和原文。
    print(f"paragraph={candidate['index']:>3} score={candidate['score']} text={candidate['text']}")  # 展示目标订单胜过四个噪声订单。

50% Needle 的 Top-3 候选：
paragraph= 50 score=9 text=权威退款记录：订单 ORDER-3 当前退款码为 RF-C43，版本 2026-07。
paragraph= 12 score=2 text=其他订单 OTHER-3-1 的退款码为 NOISE-1。
paragraph= 37 score=2 text=其他订单 OTHER-3-2 的退款码为 NOISE-2。


## 结果解读：按位置桶报告而不是只看平均值

In [4]:
retrieval_rows = []  # 收集全分块检索结果。
for document in documents:  # 逐文档检索并抽取答案。
    candidates = retrieve(document)  # 获取最高分的三个证据段。
    prediction = extract_code([candidate["text"] for candidate in candidates], document["order"])  # 从检索证据抽取目标码。
    retrieval_rows.append({"id": document["id"], "position": document["position"], "prediction": prediction, "correct": prediction == document["code"], "top_index": candidates[0]["index"]})  # 保存答案和证据位置。
print("位置桶     HeadTail  Retrieval  Top证据位置")  # 输出位置分桶表头。
for baseline, retrieval in zip(baseline_rows, retrieval_rows):  # 对齐同一 Needle 样本。
    bucket = f"{max(0, retrieval['position'] - 10):02d}-{min(100, retrieval['position'] + 10):02d}%"  # 构造可读位置桶标签。
    print(f"{bucket:<10} {str(baseline['correct']):<9} {str(retrieval['correct']):<9} {retrieval['top_index']:>5}")  # 展示中部退化和检索恢复。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算截断基线平均准确率。
retrieval_accuracy = sum(row["correct"] for row in retrieval_rows) / len(retrieval_rows)  # 计算全分块检索准确率。
middle_accuracy = sum(row["correct"] for row in baseline_rows if 20 <= row["position"] <= 80) / 3  # 单独计算中部三个位置的基线准确率。
print(f"平均准确率 {baseline_accuracy:.0%} -> {retrieval_accuracy:.0%}，HeadTail 中部准确率={middle_accuracy:.0%}")  # 防止平均值掩盖中部问题。
print("解读：这是检索管道对照，不是对真实 LLM 的性能宣称；接入模型时应保持同一数据生成器和位置分桶。")  # 明确实验可解释范围。

位置桶     HeadTail  Retrieval  Top证据位置
00-15%     True      True          5
15-35%     False     True         25
40-60%     False     True         50
65-85%     False     True         75
85-100%    True      True         95
平均准确率 40% -> 100%，HeadTail 中部准确率=0%
解读：这是检索管道对照，不是对真实 LLM 的性能宣称；接入模型时应保持同一数据生成器和位置分桶。


## 失败案例：同订单旧版本与当前版本同时命中

In [5]:
conflict_document = documents[2].copy()  # 复制中部样本以构造版本冲突。
conflict_paragraphs = list(documents[2]["paragraphs"])  # 复制段落避免修改原数据。
conflict_paragraphs[20] = f"历史退款记录：订单 {documents[2]['order']} 当前退款码为 OLD-CODE，版本 2026-06。"  # 插入同订单旧版本且也含“当前”的误导记录。
conflict_document["paragraphs"] = conflict_paragraphs  # 替换冲突文档段落。
conflict_candidates = retrieve(conflict_document, top_k=5)  # 检索多个同分版本证据。
naive_prediction = extract_code([conflict_candidates[0]["text"]], conflict_document["order"])  # 只取位置更早的 Top-1 会读旧码。
versioned_candidates = sorted(conflict_candidates, key=lambda row: ("2026-07" not in row["text"], -row["score"], row["index"]))  # 用有效版本元数据优先当前记录。
safe_prediction = extract_code([versioned_candidates[0]["text"]], conflict_document["order"])  # 从当前版本证据抽取答案。
print("冲突候选：", [(row["index"], row["score"], row["text"]) for row in conflict_candidates if documents[2]["order"] in row["text"]])  # 展示两个同订单候选。
print(f"朴素Top1={naive_prediction}，版本门禁={safe_prediction}")  # 展示陈旧值与当前值差异。
print("修正策略：语料记录携带 valid_from/source_version；同实体冲突先做时间和权威解析，再计算答案，而不是任意打破同分。")  # 总结时间语义。

冲突候选： [(20, 9, '历史退款记录：订单 ORDER-3 当前退款码为 OLD-CODE，版本 2026-06。'), (50, 9, '权威退款记录：订单 ORDER-3 当前退款码为 RF-C43，版本 2026-07。')]
朴素Top1=OLD-CODE，版本门禁=RF-C43
修正策略：语料记录携带 valid_from/source_version；同实体冲突先做时间和权威解析，再计算答案，而不是任意打破同分。


### 生产边界与评测 Manifest

In [6]:
evaluation_manifest = {"samples": len(documents), "paragraphs_per_doc": 100, "positions": positions, "distractors_per_doc": 4, "generator": "needle-r2", "metrics": ["accuracy_by_position", "evidence_index", "negative-control"]}  # 构造可复现评测清单。
print("评测 Manifest：", evaluation_manifest)  # 展示位置、干扰项和指标版本。
print("生产替换点：真实评测需多长度、多语言、随机模板、真实 Tokenizer、模型 API、多 seed、置信区间、训练污染审计和成本统计。")  # 明确固定一百段实验边界。

评测 Manifest： {'samples': 5, 'paragraphs_per_doc': 100, 'positions': [5, 25, 50, 75, 95], 'distractors_per_doc': 4, 'generator': 'needle-r2', 'metrics': ['accuracy_by_position', 'evidence_index', 'negative-control']}
生产替换点：真实评测需多长度、多语言、随机模板、真实 Tokenizer、模型 API、多 seed、置信区间、训练污染审计和成本统计。


## 回归测试：最后只保护位置覆盖、干扰项与版本冲突

In [7]:
assert [document["position"] for document in documents] == positions  # 验证五个位置桶准确生成。
assert baseline_accuracy < retrieval_accuracy and middle_accuracy == 0.0  # 验证 HeadTail 暴露中部丢失且检索对照更强。
assert retrieval_accuracy == 1.0 and all(row["top_index"] == document["position"] for row, document in zip(retrieval_rows, documents))  # 验证五个目标段均被精确定位。
assert all(sum("OTHER-" in paragraph for paragraph in document["paragraphs"]) == 4 for document in documents)  # 验证每份文档包含四个同模板干扰项。
assert naive_prediction == "OLD-CODE" and safe_prediction == documents[2]["code"]  # 验证版本冲突反例与当前版本修正。
print("回归测试通过：位置分桶、中部退化、全分块定位、四干扰项和版本冲突修正均成立。")  # 用少量断言总结长上下文评测合同。

回归测试通过：位置分桶、中部退化、全分块定位、四干扰项和版本冲突修正均成立。
